In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
CAMERA_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'figure_generation_notebooks' else Path.cwd().resolve()
EXPERIMENT_ROOT = Path('/mnt/e/watermark_rev2')
FIGURETYPE1_DIR = CAMERA_ROOT / 'figuretype1'
FIGURETYPE1_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXPERIMENT_ROOT))
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


# Estimation of the Variance Inflation Constant

Given $M$ independent sequences under the same setting, each of length $n$:

$$S_j = \sum_{t=1}^n I_{j,t}, \quad j = 1,\ldots,M$$

where $I_{j,t} = \mathbf{1}\{x_{j,t}\in G_{j,t}\}$ is the indicator variable.

**Estimators:**
- Sample mean: $\bar{S} = \frac{1}{M}\sum_{j=1}^M S_j$
- Sample variance: $\widehat{\mathrm{Var}}(S) = \frac{1}{M-1}\sum_{j=1}^M (S_j-\bar{S})^2$
- Estimated $\gamma'$: $\hat{\gamma}' = \bar{S}/n$
- Variance inflation constant: $\hat{c} = \frac{\widehat{\mathrm{Var}}(S)}{n\,\hat{\gamma}'(1-\hat{\gamma}')}$

The adjusted variance model:
$$\mathrm{Var}(S) \approx c \cdot n\,\gamma'(1-\gamma')$$

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['figure.dpi'] = 100


In [ ]:
MODEL = 'opt-125m'
DATASET = 'c4'
METHOD = 'GRID'
GAMMA = 0.5
DELTA = 10.0
N = 100
USE_NGRAM_FILTER = True


In [ ]:
def estimate_variance_inflation_constant(model, dataset, method, gamma, delta, n, use_ngram_filter=True, verbose=True):
    results_dir = EXPERIMENT_ROOT / 'results'
    result_files = sorted([f for f in results_dir.glob('*.json') if not f.name.endswith('_config.json')])
    include = []
    for f in result_files:
        if dataset in f.name and model in f.name and (method in f.name):
            include.append(f)
    if verbose:
        pass
    S_values = []
    for file_path in include:
        with open(file_path, 'r') as f:
            data = json.load(f)
        cfg_gamma = data['config']['gamma']
        cfg_delta = data['config']['delta']
        if abs(cfg_gamma - gamma) > 1e-05:
            continue
        if abs(cfg_delta - delta) > 1e-05:
            continue
        for result in data['results']:
            gt = np.array(result['watermarked']['green_token_mask'])
            ngram = result['watermarked']['ngram_info']
            ng = np.array(ngram)
            if use_ngram_filter and ngram:
                gt = gt[ng == True]
            if len(gt) < n:
                continue
            gt = gt[:n]
            S_j = np.sum(gt)
            S_values.append(S_j)
    S_values = np.array(S_values)
    M = len(S_values)
    if M < 2:
        if verbose:
            pass
        return None
    S_bar = np.mean(S_values)
    Var_S = np.var(S_values, ddof=1)
    gamma_hat = S_bar / n
    theoretical_var_iid = n * gamma_hat * (1 - gamma_hat)
    if theoretical_var_iid < 1e-10:
        if verbose:
            pass
        c_hat = np.nan
    else:
        c_hat = Var_S / theoretical_var_iid
    se_c_hat = c_hat * np.sqrt(2 / (M - 1))
    if verbose:
        pass
    return {'c_hat': c_hat, 'se_c_hat': se_c_hat, 'gamma_hat': gamma_hat, 'S_bar': S_bar, 'Var_S': Var_S, 'M': M, 'n': n, 'S_values': S_values, 'theoretical_var_iid': theoretical_var_iid}


In [ ]:
result = estimate_variance_inflation_constant(model=MODEL, dataset=DATASET, method=METHOD, gamma=GAMMA, delta=DELTA, n=N, use_ngram_filter=USE_NGRAM_FILTER, verbose=True)


In [ ]:
if result is not None:
    S_values = result['S_values']
    gamma_hat = result['gamma_hat']
    c_hat = result['c_hat']
    n = result['n']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax1 = axes[0]
    ax1.hist(S_values, bins=100, density=True, alpha=0.7, edgecolor='black', label='Empirical')
    x_range = np.linspace(S_values.min(), S_values.max(), 100)
    mu_iid = n * gamma_hat
    sigma_iid = np.sqrt(n * gamma_hat * (1 - gamma_hat))
    ax1.plot(x_range, stats.norm.pdf(x_range, mu_iid, sigma_iid), 'r--', linewidth=2, label=f'Normal (IID): c=1')
    sigma_adj = np.sqrt(c_hat * n * gamma_hat * (1 - gamma_hat))
    ax1.plot(x_range, stats.norm.pdf(x_range, mu_iid, sigma_adj), 'g-', linewidth=2, label=f'Normal (Adjusted): c={c_hat:.2f}')
    ax1.set_xlabel('$S_j$ (Sum of Green Tokens)')
    ax1.set_ylabel('Density')
    ax1.set_title(f'Distribution of $S_j$ (n={n}, M={len(S_values)})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2 = axes[1]
    z_scores = (S_values - mu_iid) / sigma_adj
    stats.probplot(z_scores, dist='norm', plot=ax2)
    ax2.set_title(f'Q-Q Plot (Standardized with c={c_hat:.2f})')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURETYPE1_DIR / f'variance_inflation_{MODEL}_{DATASET}_{METHOD}_g{GAMMA}_d{DELTA}_n{N}.pdf', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if result is not None:
    S_values = result['S_values']
    gamma_hat = result['gamma_hat']
    c_hat = result['c_hat']
    n = result['n']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax1 = axes[0]
    ax1.hist(S_values, bins=30, density=True, alpha=0.7, edgecolor='black', label='Empirical')
    x_range = np.linspace(S_values.min(), S_values.max(), 100)
    mu_iid = n * gamma_hat
    sigma_iid = np.sqrt(n * gamma_hat * (1 - gamma_hat))
    ax1.plot(x_range, stats.norm.pdf(x_range, mu_iid, sigma_iid), 'r--', linewidth=2, label=f'Normal (IID): c=1')
    sigma_adj = np.sqrt(c_hat * n * gamma_hat * (1 - gamma_hat))
    ax1.plot(x_range, stats.norm.pdf(x_range, mu_iid, sigma_adj), 'g-', linewidth=2, label=f'Normal (Adjusted): c={c_hat:.2f}')
    ax1.set_xlabel('$S_j$ (Sum of Green Tokens)')
    ax1.set_ylabel('Density')
    ax1.set_title(f'Distribution of $S_j$ (n={n}, M={len(S_values)})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2 = axes[1]
    z_scores = (S_values - mu_iid) / sigma_adj
    stats.probplot(z_scores, dist='norm')
    ax2.set_title(f'Q-Q Plot (Standardized with c={c_hat:.2f})')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURETYPE1_DIR / f'variance_inflation_{MODEL}_{DATASET}_{METHOD}_g{GAMMA}_d{DELTA}_n{N}.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    r = stats.probplot(z_scores, dist='norm')
    slope, intercept, r = r[1]


In [ ]:
def batch_estimate_c(models, datasets, methods, gammas, deltas, lengths):
    results = []
    for model in models:
        for dataset in datasets:
            for method in methods:
                for gamma in gammas:
                    for delta in deltas:
                        for n in lengths:
                            res = estimate_variance_inflation_constant(model=model, dataset=dataset, method=method, gamma=gamma, delta=delta, n=n, verbose=False)
                            if res is not None:
                                results.append({'model': model, 'dataset': dataset, 'method': method, 'gamma': gamma, 'delta': delta, 'n': n, 'M': res['M'], 'gamma_hat': res['gamma_hat'], 'c_hat': res['c_hat'], 'se_c_hat': res['se_c_hat'], 'Var_S': res['Var_S'], 'theoretical_var_iid': res['theoretical_var_iid']})
    return pd.DataFrame(results)


In [ ]:
MODELS = ['gpt2', 'opt-125m', 'pythia-160m']
DATASETS = ['c4', 'lfqa', 'wikipedia']
METHODS = ['GRID']
GAMMAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
DELTAS = [0.5, 1.0]
LENGTHS = [30, 50, 100]
df_results = batch_estimate_c(MODELS, DATASETS, METHODS, GAMMAS, DELTAS, LENGTHS)
if len(df_results) > 0:
    pass
else:
    pass


In [ ]:
df_results['c_hat']


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
df = df_results[(df_results['gamma'] == 0.2) & (df_results['delta'] == 1)].copy()
df['c_hat'] = pd.to_numeric(df['c_hat'], errors='coerce')
df = df.dropna(subset=['c_hat'])
LABEL_FSIZE = 14
TICK_FSIZE = 12
sns.set_theme(style='white', context='paper')
plt.rcParams.update({'axes.labelsize': LABEL_FSIZE, 'legend.fontsize': LABEL_FSIZE, 'legend.title_fontsize': LABEL_FSIZE, 'xtick.labelsize': TICK_FSIZE, 'ytick.labelsize': TICK_FSIZE, 'axes.linewidth': 1.2})

def draw_kde_panel(ax, split_var):
    levels = sorted(df[split_var].dropna().unique())
    palette = sns.color_palette('Set2', len(levels))
    color_map = dict(zip(levels, palette))
    sns.kdeplot(data=df, x='c_hat', hue=split_var, fill=True, common_norm=False, alpha=0.28, linewidth=2, palette=color_map, ax=ax, legend=False)
    legend_elements = []
    for value, group in df.groupby(split_var, dropna=True):
        mean_val = group['c_hat'].mean()
        color = color_map[value]
        ax.axvline(mean_val, linestyle='--', linewidth=2, color=color)
        legend_elements.append(Line2D([0], [0], color=color, lw=3, label=f'{mean_val:.2f}  {str(value).upper()}'))
    ax.set_xlabel('$\\hat{c}$')
    ax.set_ylabel('')
    ax.yaxis.set_major_locator(MaxNLocator(3))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    ax.grid(False)
    ax.tick_params(axis='both', which='major', width=1.0, length=4)
    ax.legend(handles=legend_elements, title=split_var.title(), frameon=False, loc='best')
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
draw_kde_panel(axes[0], 'model')
draw_kde_panel(axes[1], 'dataset')
fig.text(0.02, 0.5, 'Density', va='center', rotation='vertical', fontsize=LABEL_FSIZE)
plt.tight_layout(rect=[0.05, 0, 1, 1])
plt.savefig(FIGURETYPE1_DIR / 'kde_c_hat_model_dataset_gamma0.2_delta1.pdf', dpi=300, bbox_inches='tight')
plt.show()
for split_var in ['n', 'gamma', 'delta', 'dataset']:
    fig, ax = plt.subplots(figsize=(7.5, 5))
    draw_kde_panel(ax, split_var)
    fig.text(0.02, 0.5, 'Density', va='center', rotation='vertical', fontsize=LABEL_FSIZE)
    plt.tight_layout(rect=[0.05, 0, 1, 1])
    plt.savefig(FIGURETYPE1_DIR / f'kde_c_hat_by_{split_var}_gamma0.2_delta1.pdf', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
MODELS_FULL = ['opt-125m']
DATASETS_FULL = ['c4', 'lfqa', 'wikipedia']
METHODS_FULL = ['GRID']
GAMMAS_FULL = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
DELTAS_FULL = [0.5, 1.0, 2, 5, 10]
LENGTHS_FULL = [50, 100]
df_full = batch_estimate_c(MODELS_FULL, DATASETS_FULL, METHODS_FULL, GAMMAS_FULL, DELTAS_FULL, LENGTHS_FULL)
if len(df_full) > 0:
    df_full.to_csv(FIGURETYPE1_DIR / 'variance_inflation_comprehensive.csv', index=False)
else:
    pass


In [ ]:
GAMMAS_FULL


In [ ]:
df = df_full[df_full['dataset'] != 'wikipedia']


In [ ]:
np.nanmean(df[df['delta'] <= 1]['c_hat'])


In [ ]:
np.nanmean(df_full['c_hat'])


In [ ]:
MODELS_FULL = ['opt-125m']
DATASETS_FULL = ['c4', 'lfqa', 'wikipedia']
METHODS_FULL = ['QQ']
GAMMAS_FULL = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
DELTAS_FULL = [0.5, 1.0, 2, 5, 10]
LENGTHS_FULL = [500]
df_qq = batch_estimate_c(MODELS_FULL, DATASETS_FULL, METHODS_FULL, GAMMAS_FULL, DELTAS_FULL, LENGTHS_FULL)


In [ ]:
if len(df_full) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    ax = axes[0, 0]
    df_full.boxplot(column='c_hat', by='gamma', ax=ax)
    ax.set_xlabel('gamma')
    ax.set_ylabel('c_hat')
    ax.set_title('c_hat by gamma')
    ax.axhline(y=1, color='r', linestyle='--', alpha=0.5)
    ax = axes[0, 1]
    df_full.boxplot(column='c_hat', by='delta', ax=ax)
    ax.set_xlabel('delta')
    ax.set_ylabel('c_hat')
    ax.set_title('c_hat by delta')
    ax.axhline(y=1, color='r', linestyle='--', alpha=0.5)
    ax = axes[0, 2]
    df_full.boxplot(column='c_hat', by='n', ax=ax)
    ax.set_xlabel('n (sequence length)')
    ax.set_ylabel('c_hat')
    ax.set_title('c_hat by sequence length')
    ax.axhline(y=1, color='r', linestyle='--', alpha=0.5)
    ax = axes[1, 0]
    df_full.boxplot(column='c_hat', by='method', ax=ax)
    ax.set_xlabel('method')
    ax.set_ylabel('c_hat')
    ax.set_title('c_hat by method')
    ax.axhline(y=1, color='r', linestyle='--', alpha=0.5)
    ax = axes[1, 1]
    df_full.boxplot(column='c_hat', by='model', ax=ax)
    ax.set_xlabel('model')
    ax.set_ylabel('c_hat')
    ax.set_title('c_hat by model')
    ax.axhline(y=1, color='r', linestyle='--', alpha=0.5)
    ax = axes[1, 2]
    df_full.boxplot(column='c_hat', by='dataset', ax=ax)
    ax.set_xlabel('dataset')
    ax.set_ylabel('c_hat')
    ax.set_title('c_hat by dataset')
    ax.axhline(y=1, color='r', linestyle='--', alpha=0.5)
    plt.suptitle('Variance Inflation Constant by Different Factors', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURETYPE1_DIR / 'variance_inflation_by_factors.pdf', dpi=150, bbox_inches='tight')
    plt.show()
